# Retail Sales & Profitability Analytics: Exploratory Data Analysis
**Role:** Senior Data Analyst & Analytics Engineer  
**Project:** Final-Year B.Tech CSE Placement Portfolio Project  
**Dataset:** Tableau Sample Superstore Dataset (9,994 transactions, 2014-2017)  
**Core Stack:** Python 3, pandas, NumPy, SQLite, Matplotlib, Seaborn, openpyxl  

---
## 1. Executive Context & Analytical Objectives
This notebook provides end-to-end exploratory analysis and data engineering for an enterprise retail business. Management has observed top-line revenue growth across four fiscal years (2014–2017), but net margins remain compressed (~12.5%). Our goal is to uncover:
- Which product subcategories and items drive profits versus bleed red ink.
- The quantitative relationship between promotional discounting and profit destruction.
- Regional, state, and customer segment dynamics.
- Evidence-based recommendations to restore sustainable margin health.


In [1]:
import sys
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans', 'Arial']
pd.set_option('display.max_columns', None)

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.config import RAW_DATA_FILE, DATABASE_FILE
from src.data_ingestion import load_raw_data
from src.data_cleaning import clean_retail_data
from src.feature_engineering import engineer_features
from src.database import get_db_connection, execute_query
from src.metrics import calculate_kpi_summary, calculate_category_summary, calculate_discount_sensitivity

print('Environment initialized and dependencies loaded successfully.')

## 2. Raw Dataset Ingestion & Profiling
We load the raw CSV dataset and inspect column structures, non-null counts, and memory usage.

In [2]:
raw_df = load_raw_data(RAW_DATA_FILE)
print(f'Raw Dataset Shape: {raw_df.shape[0]:,} rows by {raw_df.shape[1]} columns')
raw_df.head(3)

## 3. Data Cleaning, Validation & Quarantine Pipeline
Rather than silently dropping non-compliant rows, our pipeline enforces strict validation rules:
1. Standardizes columns to `snake_case`.
2. Validates ISO date parsing.
3. Verifies chronological sanity (`ship_date >= order_date`).
4. Checks financial boundaries (`sales > 0`, `quantity >= 1`, `0.0 <= discount <= 1.0`).
5. Zero-pads postal codes to 5-digit strings.
6. Quarantines invalid records into `retail_sales_rejected.csv` with explicit rejection reasons.

In [3]:
cleaned_df, rejected_df, quality_metrics = clean_retail_data(raw_df)
enriched_df = engineer_features(cleaned_df)
print('Data Quality Report Summary:')
for k, v in quality_metrics.items():
    if k != 'rejection_reasons_breakdown':
        print(f'  - {k}: {v}')

## 4. Executive KPI Scorecard
Computing high-level enterprise measures across the full 4-year transactional history.

In [4]:
kpi_summary = calculate_kpi_summary(enriched_df)
kpi_display = pd.DataFrame([kpi_summary]).T
kpi_display.columns = ['Enterprise Value']
kpi_display

## 5. SQL Analytics via SQLite Database Engine
We query the embedded database `retail_analytics.db` to demonstrate portable SQL analytics.

In [5]:
# Query: Category & Subcategory Profitability Breakdown
query_subcat = '''
SELECT
    category,
    sub_category,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100.0, 2) AS profit_margin_pct,
    SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) AS loss_making_lines
FROM retail_orders
GROUP BY category, sub_category
ORDER BY category ASC, total_profit DESC;
'''
df_subcat = execute_query(query_subcat)
df_subcat

## 6. Deep Dive: Discount Sensitivity & Margin Destruction
Let us evaluate how promotional discounting correlates with bottom-line profitability.

In [6]:
query_disc = '''
SELECT
    discount_tier,
    COUNT(*) AS order_lines,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit,
    ROUND((SUM(profit) / SUM(sales)) * 100.0, 2) AS realized_margin_pct,
    ROUND(100.0 * SUM(CASE WHEN profit < 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS loss_order_pct
FROM retail_orders
GROUP BY discount_tier
ORDER BY MIN(discount) ASC;
'''
df_disc = execute_query(query_disc)
df_disc

### Visualizing Margin Erosion Across Discount Tiers
The chart below illustrates the tipping point beyond which discounting destroys business value.

In [7]:
fig, ax1 = plt.subplots(figsize=(10, 5))
x = range(len(df_disc))
colors = ['#059669' if m >= 0 else '#DC2626' for m in df_disc['realized_margin_pct']]
bars = ax1.bar(x, df_disc['realized_margin_pct'], color=colors, width=0.5, alpha=0.85, label='Profit Margin %')
ax1.axhline(0, color='#475569', linewidth=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(df_disc['discount_tier'], rotation=15, ha='right', weight='bold')
ax1.set_ylabel('Realized Margin (%)', weight='bold')

ax2 = ax1.twinx()
ax2.plot(x, df_disc['loss_order_pct'], color='#E11D48', marker='o', linewidth=2.5, label='Loss-Making Lines %')
ax2.set_ylabel('Loss-Making Transactions (%)', color='#E11D48', weight='bold')
ax2.grid(False)

plt.title('Discount Margin Erosion: The 20% Discount Cliff', fontsize=13, weight='bold', pad=12)
plt.tight_layout()
plt.show()

## 7. Product Catalog: Identifying "Sales Traps"
We isolate products with significant sales volume (>= $2,500) that generate net losses for the firm.

In [8]:
query_traps = '''
SELECT
    product_name,
    category,
    sub_category,
    ROUND(SUM(sales), 2) AS sales,
    ROUND(SUM(profit), 2) AS net_loss,
    ROUND((SUM(profit) / SUM(sales)) * 100.0, 2) AS margin_pct,
    ROUND(AVG(discount) * 100.0, 1) AS avg_discount_pct
FROM retail_orders
GROUP BY product_id, product_name, category, sub_category
HAVING SUM(sales) >= 2500.0 AND SUM(profit) < 0
ORDER BY net_loss ASC;
'''
df_traps = execute_query(query_traps)
df_traps

## 8. Summary of Findings & Evidence-Based Recommendations
1. **The 20% Discount Ceiling:** Transactions at 0% discount earn ~30% margin. When discounts exceed 20%, margins plunge into severe negative territory (-14% to -83%). Management must institute a hard approval workflow for discounts above 20%.
2. **Restructure Loss-Leader Subcategories:** Tables (-$17.7K net loss) and Bookcases (-$3.5K net loss) suffer structural losses driven by deep promotions. Re-negotiate supplier cost contracts, bundle with high-margin accessories, or impose minimum selling price floors.
3. **Regional Strategy:** The West and East regions contribute over 75% of cumulative profit, while Central experiences lower profitability due to higher promotional depth in Illinois and Texas.
4. **Customer Retention:** Corporate and Consumer segments drive the core volume, but retaining high-margin accounts through targeted loyalty incentives yields significantly higher return on sales than broad promotional discounting.
